# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fawadwazir/flyrank-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

Deployed paper: **[live URL — see `submission/paper_url.txt`]** &nbsp;|&nbsp; Full write-up: `work/capstone_report.md`

## 1. Question

Which of a client's existing pages should an editor refresh first, out of thousands that
could plausibly qualify? This asks for a **ranking**, not a single yes/no — the decision it
supports is: given limited editor time, which pages go on this sprint's worklist first.

In [1]:
# No computation needed for this section — see write-up above and paper §1.
print("Lane 2 — Refresh / Content Opportunity Scoring")

Lane 2 — Refresh / Content Opportunity Scoring


## 2. Data

`data/raw/content_refresh_anonymized.csv` — 30,000 pages, 32 clients, 52 columns
(identifiers, content metadata, 90-day performance, pre-computed trend/tier fields). All
IDs are pre-anonymized hashes.

The data contract (`w03_data_contract.ipynb`) scoped a label definition against the full
79M-row `FlyRank/internship-warehouse` on Hugging Face. This capstone iteration runs on the
30,000-row starter slice — the same one the baseline was validated on — so the comparison in
§4 is apples-to-apples. Scaling to the full warehouse is future work (paper §7).

**Excluded on purpose:** `trend_direction`/`trend_pct` (the label itself), the paired 30-day
windows the label is computed from, and both ID columns (grouping only, never features).

In [2]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print("Date-window fields:", [c for c in df.columns if "90d" in c or "30d" in c][:6], "...")

30,000 rows, 32 clients, 44 columns
Date-window fields: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d'] ...


## 3. Methodology

- **Label:** `is_declining_label = 1` if `trend_direction == "down"` — a proxy, not a
  guarantee of a future outcome.
- **Baseline:** transparent 3-gate rule (stale + strikable + visible), ranked by raw
  impressions — built in `w04_baseline_score.ipynb`.
- **Features:** 18 numeric (standardized) + 8 categorical (one-hot) = 26, listed in full in
  `w05_model.ipynb` and paper §3.
- **Models:** Logistic Regression and Random Forest, same features, same splits.
- **Validation design:** `GroupShuffleSplit` grouped by `client_id` (75/25, seed 42) — a
  client-holdout, not a page-level random split, so the model is scored on clients it never
  trained on.
- **Leakage checks:** an `assert` blocks the label, the label's raw components, and both ID
  columns from the feature list (`w05_model.ipynb`); a random-split comparison
  (`w06_validation_audit.ipynb`) confirms the client-holdout split materially changes the
  score, i.e. it's catching real leakage risk, not a paper exercise.

In [3]:
import json
r = json.load(open("../artifacts/model_results.json"))
print(f"Train: {r['n_train']:,} rows / {r['n_train_clients']} clients")
print(f"Test:  {r['n_test']:,} rows / {r['n_test_clients']} clients")
print(f"Base rate  train={r['train_base_rate']:.3f}  test={r['test_base_rate']:.3f}")

Train: 22,885 rows / 24 clients
Test:  7,115 rows / 8 clients
Base rate  train=0.550  test=0.517


## 4. Results (vs baseline)

Model vs. baseline, same client-holdout test set, same metrics:

| Method | ROC AUC | P@20 | P@50 |
|---|---|---|---|
| Baseline rule | 0.503 | 0.20 | 0.34 |
| Random Forest | 0.603 | 0.50 | 0.56 |
| **Logistic Regression** | **0.611** | **0.80** | **0.74** |

Logistic Regression roughly doubles the baseline's Precision@50. The split-honesty check
(`w06`) shows the same Random Forest scores 0.753 AUC / 0.92 P@50 on a random split instead
of client-holdout — that gap is the paper's central finding, not a better model.

In [4]:
print("Baseline: ", r["baseline_rule"])
print("Random Forest (honest split):", r["models"]["random_forest"])
print("Logistic Regression (honest split):", r["models"]["logistic_regression"])
v = json.load(open("../artifacts/validation_audit.json"))
print("\nRandom Forest, LEAKY random split:", v["random_split_check"])

Baseline:  {'roc_auc': 0.5026313874386964, 'precision_at_20': 0.2, 'precision_at_50': 0.34, 'pct_flagged': 0.052986647926914966}
Random Forest (honest split): {'roc_auc': 0.6028772346147762, 'precision_at_20': 0.5, 'precision_at_50': 0.56}
Logistic Regression (honest split): {'roc_auc': 0.6111010124980224, 'precision_at_20': 0.8, 'precision_at_50': 0.74}

Random Forest, LEAKY random split: {'roc_auc': 0.7525183740920378, 'precision_at_50': 0.92, 'test_base_rate': 0.542}


## 5. Limitations

- **Proxy label, not ground truth** — reflects a computed trend, not a verified cause.
- **Directional, not causal** — association with decline, not proof that refreshing *causes*
  recovery; that needs a pre/post or holdout experiment.
- **Small client pool** — 32 clients (24 train / 8 test) is thin for a client-holdout
  evaluation; an early signal, not a settled number, until validated at warehouse scale.
- **Per-client concentration** — both baseline and model over-represent 2–4 clients at the
  top of the queue (see §6 below).
- **Complexity didn't help** — Random Forest underperformed the simpler linear model here;
  reported as a genuine finding, not tuned away.

In [5]:
e = json.load(open("../artifacts/error_analysis.json"))
print(f"Logistic Regression top-50 precision: {e['top50_true_positives']}/50")
print("Client concentration in top-50:", e["top50_client_counts"])

Logistic Regression top-50 precision: 37/50
Client concentration in top-50: {'client_f369cb89fc': 22, 'client_d029fa3a95': 13, 'client_4e07408562': 8, 'client_8527a891e2': 7}


## 6. Ranked recommendations

1. Adopt the Logistic Regression queue as the refresh worklist, reviewed by a human before
   any page is touched.
2. Cap how many pages any single client contributes to the top of the queue — a per-client
   quota, not just a global sort.
3. Work `refresh_priority` this sprint, `refresh_backlog` next sprint, skip `no_action`.
4. Re-validate quarterly as search behavior and competitive tiers shift.
5. Never quote Precision@K on this dataset without stating the split type.

In [6]:
p = json.load(open("../artifacts/playbook_summary.json"))
print("Queue tier sizes:", p["action_counts"])
q = pd.read_csv("../artifacts/model_ranked_queue_test.csv")
q.head(10)[["content_id","client_id","model_score","action","reason_code","is_declining_label"]]

Queue tier sizes: {'no_action': 4269, 'refresh_backlog': 2134, 'refresh_priority': 712}


,content_id,client_id,model_score,action,reason_code,is_declining_label
0,content_a928cb66d230,client_f369cb89fc,0.971268,refresh_priority,strikable_position,1
1,content_7be5f150dc65,client_f369cb89fc,0.966379,refresh_priority,strikable_position,0
2,content_a8864e189b2e,client_d029fa3a95,0.965735,refresh_priority,strikable_position,1
3,content_5d77d3077984,client_f369cb89fc,0.959252,refresh_priority,strikable_position+inconsistent_visibility,1
4,content_87c007fb5c26,client_f369cb89fc,0.952377,refresh_priority,strikable_position+visible,1
5,content_b5e9e6453511,client_f369cb89fc,0.951314,refresh_priority,strikable_position,1
6,content_c82bc0c24241,client_f369cb89fc,0.949972,refresh_priority,strikable_position+visible,1
7,content_ff102de380d8,client_f369cb89fc,0.947525,refresh_priority,strikable_position,1
8,content_96dba8ca02c1,client_f369cb89fc,0.946452,refresh_priority,strikable_position,1
9,content_374e795aab68,client_f369cb89fc,0.945569,refresh_priority,low_signal,0


## 7. Artifacts the paper embeds

The five charts embedded in the deployed paper, generated from the real results above.

In [7]:
import sys
sys.path.insert(0, "../../scripts")
from ml_utils import simple_svg_bar_chart
from pathlib import Path
Path("../artifacts/charts").mkdir(parents=True, exist_ok=True)

simple_svg_bar_chart("Precision@50 \u2014 client-holdout test set",
    ["Baseline rule", "Random Forest", "Logistic Regression"],
    [r["baseline_rule"]["precision_at_50"], r["models"]["random_forest"]["precision_at_50"],
     r["models"]["logistic_regression"]["precision_at_50"]],
    Path("../artifacts/charts/precision_at_50_comparison.svg"), color="#4C6EF5")

simple_svg_bar_chart("Precision@20 \u2014 client-holdout test set",
    ["Baseline rule", "Random Forest", "Logistic Regression"],
    [r["baseline_rule"]["precision_at_20"], r["models"]["random_forest"]["precision_at_20"],
     r["models"]["logistic_regression"]["precision_at_20"]],
    Path("../artifacts/charts/precision_at_20_comparison.svg"), color="#7048E8")

simple_svg_bar_chart("Random Forest ROC AUC \u2014 split honesty check",
    ["Client-holdout (honest)", "Random split (leaky)"],
    [r["models"]["random_forest"]["roc_auc"], v["random_split_check"]["roc_auc"]],
    Path("../artifacts/charts/split_honesty_gap.svg"), color="#E8590C")

top_coefs = sorted(e["top_lr_coefs"], key=lambda x: -abs(x[1]))[:6]
simple_svg_bar_chart("Logistic Regression \u2014 top standardized coefficients",
    [n.replace("cat__","").replace("num__","") for n,_ in top_coefs],
    [abs(c) for _,c in top_coefs],
    Path("../artifacts/charts/lr_top_coefficients.svg"), color="#2F9E44")

top_imp = sorted(e["top_rf_importances"], key=lambda x: -x[1])[:8]
simple_svg_bar_chart("Random Forest \u2014 top 8 feature importances",
    [n.replace("cat__","").replace("num__","") for n,_ in top_imp],
    [v for _,v in top_imp],
    Path("../artifacts/charts/rf_feature_importance.svg"), color="#2F9E44")

print(sorted(p.name for p in Path("../artifacts/charts").glob("*.svg")))

['lr_top_coefficients.svg', 'precision_at_20_comparison.svg', 'precision_at_50_comparison.svg', 'rf_feature_importance.svg', 'split_honesty_gap.svg']


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only anonymized hash IDs
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/` — then submit repo URL on the card
- [ ] Deployed paper has all 9 sections including Abstract (top) and Acknowledgments &
  data credit (bottom, linking https://flyrank.ai)